# Lab 3 : analyse des résultats

Ce notebook lit les CSV du dossier `results/`. Ces fichiers sont produits par GitHub Actions (workflow `experiments.yml`), sur 3 machines : Ubuntu x86, Ubuntu ARM et macOS ARM.

Le notebook ne lance aucune expérience, il lit juste les résultats.

Pour le relancer : `pip install -r requirements-notebook.txt`, puis *Restart Kernel and Run All Cells*.

In [ ]:
import glob

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.tree import DecisionTreeRegressor, plot_tree

pd.set_option("display.precision", 2)

df = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("results/results_*.csv"))], ignore_index=True)
causal = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("results/causal_*.csv"))], ignore_index=True)

# une colonne pour savoir sur quelle machine ça a tourné
df["machine"] = df["os"] + " " + df["arch"]
causal["machine"] = causal["os"] + " " + causal["arch"]
df["Compilateur"] = df["Compilateur"].fillna("aucun")

FACTEURS = ["Repetitions", "Language", "Propriete", "Librairie", "Precision", "Intervalle", "Compilateur", "machine"]
print(len(df), "exécutions,", df["config_id"].nunique(), "configurations")
df.head()

## 1. Ce qu'on a lancé

Notre feature model (`fm/experiment.uvl`) a 192 configurations. On en a tiré 60 au hasard (seed 1), et on a ajouté nos 4 programmes des labs 1 et 2. Ça fait 64 configurations.

Chaque configuration est lancée 5 fois, parce que le résultat est aléatoire. Les lignes `Ubuntu` tournent sur les deux runners Ubuntu (x86 et ARM), les lignes `MacOS` sur le runner macOS.

In [ ]:
df.groupby("machine").agg(configurations=("config_id", "nunique"), executions=("percentage", "size"))

## 2. Est-ce que le résultat bouge d'une exécution à l'autre ?

Oui. On regarde l'écart-type entre les 5 exécutions d'une même configuration, selon le nombre de répétitions.

In [ ]:
ecart = (df.groupby(["config_id", "machine", "Repetitions"])["percentage"].std()
           .groupby("Repetitions").agg(["mean", "max"]))
ecart.columns = ["écart-type moyen", "écart-type max"]
ecart

Plus il y a de répétitions, moins le résultat bouge. Avec 10 000 répétitions le pourcentage varie d'environ 0,5 point entre deux exécutions, avec 1 000 000 il ne bouge presque plus.

C'est normal : on estime une proportion, donc l'erreur diminue comme 1/√n. Le nombre de répétitions ne change pas la valeur du résultat, il change juste sa précision.

Donc si on relance le workflow sur le même commit, on n'a pas exactement les mêmes chiffres (les tirages ne sont pas fixés par une graine), mais on reste dans cet écart.

## 3. Effet de chaque facteur

Pour chaque facteur, la moyenne du pourcentage selon sa valeur. On sépare les deux propriétés parce qu'elles n'ont pas du tout le même niveau.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharey=True)
for ax, facteur in zip(axes.flat, FACTEURS):
    if facteur == "Propriete":
        df.groupby("Propriete")["percentage"].mean().plot.bar(ax=ax, color="gray")
    else:
        df.groupby([facteur, "Propriete"])["percentage"].mean().unstack().plot.bar(ax=ax)
    ax.set_title(facteur)
    ax.set_xlabel("")
    ax.set_ylim(55, 90)
    ax.tick_params(axis="x", rotation=30)
axes[0, 0].set_ylabel("% de cas où la propriété est vraie")
axes[1, 0].set_ylabel("% de cas où la propriété est vraie")
plt.tight_layout()
plt.show()

In [ ]:
for facteur in FACTEURS:
    if facteur != "Propriete":
        display(df.groupby([facteur, "Propriete"])["percentage"].agg(["mean", "std", "count"]).unstack("Propriete"))

Ce qu'on voit :

- **La propriété** a le plus gros effet : l'associativité est vraie plus souvent que la distributivité.
- **La précision et l'intervalle** ont un gros effet aussi : en `float`, ou avec des nombres entre 0 et 100, le pourcentage baisse.
- **Le langage** a l'air d'avoir un effet (le C est plus bas), mais on va voir que ce n'est pas vraiment le langage.
- **Le nombre de répétitions et la librairie** ne changent presque rien à la moyenne. Pareil pour gcc ou clang (la colonne `aucun`, c'est Python et JavaScript).
- **La machine** ne change rien, sauf dans un cas précis (partie 6).

## 4. Arbre de décision

Pour vérifier ça de façon plus systématique, on entraîne un arbre de décision qui prédit le pourcentage à partir des facteurs. Les facteurs qu'il utilise en premier sont ceux qui comptent le plus.

In [ ]:
X = pd.get_dummies(df[FACTEURS].astype(str))
arbre = DecisionTreeRegressor(max_depth=4, random_state=0).fit(X, df["percentage"])

plt.figure(figsize=(22, 9))
plot_tree(arbre, feature_names=list(X.columns), filled=True, rounded=True, precision=1, fontsize=8)
plt.show()

importance = pd.Series(arbre.feature_importances_, index=X.columns)
importance.groupby(lambda nom: nom.split("_")[0]).sum().sort_values(ascending=False).round(3).to_frame("importance")

L'arbre confirme : il coupe d'abord sur la propriété, puis sur la façon dont les nombres sont fabriqués (précision, intervalle, langage). Les répétitions ne servent presque pas.

## 5. Le vrai facteur : comment les nombres sont fabriqués

En regardant les tableaux, on a remarqué qu'il n'y a en fait que deux niveaux pour chaque propriété. On a une explication.

`random.random()` en Python et `Math.random()` en JavaScript tirent un entier de 53 bits et le divisent par 2^53. Les nombres obtenus sont tous des multiples de 2^-53 : ils sont sur une grille. Un petit nombre sur cette grille a des zéros à la fin de sa mantisse, donc les additions tombent plus souvent juste, sans arrondi.

Dans les autres cas, la mantisse est remplie jusqu'au dernier bit :

- en C, `rand() / RAND_MAX` divise par 2^31 - 1, qui n'est pas une puissance de 2 ;
- quand on multiplie par 100 pour avoir des nombres entre 0 et 100 ;
- en `float`, parce qu'on arrondit à 24 bits.

Là il y a plus d'arrondis, donc la propriété est vraie moins souvent.

On range chaque configuration dans un des deux groupes :

In [ ]:
grille = (df["Precision"] == "double") & (df["Intervalle"] == "Max1") & (df["Language"] != "C")
df["nombres"] = grille.map({True: "sur la grille 2^-53", False: "mantisse pleine"})

df.groupby(["Propriete", "nombres"])["percentage"].agg(["mean", "std", "min", "max", "count"])

À l'intérieur d'un groupe, le pourcentage ne bouge presque plus, quel que soit le langage ou la librairie. Donc le langage n'est pas la cause : c'est juste que le C n'a pas le même générateur.

La seule exception est la distributivité en C sur ARM, qu'on regarde juste après.

### Ça explique nos résultats des labs 1 et 2

In [ ]:
labs = pd.DataFrame({
    "programme": ["C (lab 1)", "Python (lab 1)", "JavaScript ChatGPT (lab 2)", "Python Gemini (lab 3)"],
    "config_id": ["tp1.c", "tp1.python", "llm.chatgpt.js", "llm.gemini.python"],
    "résultat d'origine": [75, 82.54, 74.82, 82.75],
})
ancres = df[df["origin_of_config"] == "anchor"].groupby("config_id").agg(
    nombres=("nombres", "first"), refait_ici=("percentage", "mean"))
labs.merge(ancres, on="config_id")

On retrouve les valeurs d'origine. Et on comprend l'écart entre nos programmes :

- notre C du lab 1 est en `float` entre 0 et 100, donc mantisse pleine : environ 75 % ;
- le JavaScript de ChatGPT multiplie par 100, donc mantisse pleine aussi : environ 75 % ;
- les deux programmes Python utilisent `random()` directement, donc la grille : environ 83 %.

Ce n'était pas une histoire de langage ou d'OS, mais de comment chaque programme tire ses nombres.

## 6. Effet de la machine

On compare les 3 runners. Ubuntu x86 et Ubuntu ARM ont le même OS mais pas le même processeur. Ubuntu ARM et macOS ont le même type de processeur mais pas le même OS.

In [ ]:
df["cas"] = df["Language"].where(df["Language"] == "C", "Python / JavaScript")
df.pivot_table(index=["Propriete", "nombres", "cas"], columns="machine", values="percentage", aggfunc="mean")

Tout est pareil sur les 3 machines, sauf une ligne : la distributivité en C. Sur x86 on a environ 64,5 %, sur ARM (Linux et macOS) environ 69 %.

Ce n'est donc pas l'OS, c'est le processeur. Notre explication : sur ARM, le compilateur utilise l'instruction FMA, qui calcule `x * y + x * z` avec un seul arrondi au lieu de deux. Le résultat n'est plus le même, donc la comparaison non plus. Python et JavaScript ne sont pas touchés, parce qu'ils font chaque opération séparément.

## 7. Corrélation ou causalité ? Deux expériences pour vérifier

Jusqu'ici on a des corrélations. Pour montrer que nos deux explications sont bien les causes, on a écrit `causal.py`. Il change une seule chose à la fois et garde tout le reste identique. Il tourne aussi dans GitHub Actions.

### Expérience 1 : la mantisse

Même programme Python, associativité, 100 000 répétitions. On change seulement la façon de tirer un nombre.

In [ ]:
mantisse = causal[causal["experience"] == "mantisse"]
mantisse.pivot_table(index="variante", columns="machine", values="percentage", aggfunc="mean")

- `random()` : la grille, environ 83 %.
- `random() * 100` : on tombe à environ 75 %.
- `randrange(2**31) / (2**31 - 1)` : on imite le `rand() / RAND_MAX` du C, et on retrouve le résultat du C (environ 74,5 %) alors qu'on est en Python.
- `randrange(2**31) / 2**31` : même tirage, mais on divise par une puissance de 2. Les nombres n'ont que 31 bits utiles, les additions sont exactes, et on obtient 100 %.

Entre les deux dernières lignes, la seule différence est le diviseur. C'est bien le nombre de bits utiles qui cause l'écart, pas le langage.

### Expérience 2 : le FMA

Même programme C, distributivité. On change seulement les options de compilation. `-ffp-contract=off` interdit au compilateur de fusionner la multiplication et l'addition. Sur x86, `-mfma` l'autorise à utiliser l'instruction FMA.

In [ ]:
fma = causal[causal["experience"] == "fma"]
display(fma.pivot_table(index="variante", columns="machine", values="percentage", aggfunc="mean"))
fma.groupby(["machine", "variante"])["compiler_version"].first().to_frame()

Sur ARM, dès qu'on met `-ffp-contract=off`, on retombe sur la valeur du x86 (environ 64,5 %). Et sur x86, si on active le FMA avec `-mfma`, on monte à la valeur de l'ARM (environ 69 %).

Donc la cause est bien le FMA, et pas le processeur en lui-même ni l'OS.

## 8. Le compilateur

Dans l'échantillon, gcc et clang donnent les mêmes résultats. Mais attention au tableau ci-dessous : sur macOS, la commande `gcc` lance en réalité Apple clang. Sur ce runner on n'a donc jamais vraiment testé gcc.

In [ ]:
c = df[df["Language"] == "C"]
c.groupby(["machine", "Compilateur"]).agg(version=("compiler_version", "first"), moyenne=("percentage", "mean"))

## 9. Nos recommandations pour avoir un résultat stable

- **Dire quelle propriété on teste**, c'est ce qui change le plus le résultat.
- **Dire comment on tire les nombres** : générateur, intervalle, `float` ou `double`. C'est ça qui explique l'écart entre 75 % et 83 % dans nos labs.
- **Au moins 100 000 répétitions**, mieux 1 000 000 : le résultat bouge de moins de 0,1 point.
- **En C, compiler avec `-ffp-contract=off`** pour avoir le même résultat sur x86 et sur ARM.
- **Fixer la graine** si on veut exactement le même chiffre à chaque fois. On ne l'a pas fait ici.
- Le langage, la librairie, l'OS et le compilateur n'ont pas d'effet à eux seuls, une fois que le reste est fixé.

## 10. Limites

- La graine de chaque exécution n'est pas enregistrée. On peut relancer une configuration, mais pas retrouver exactement les mêmes tirages.
- Sur macOS, `gcc` est en fait clang.
- JavaScript est peu présent dans l'échantillon (5 configurations), parce qu'il a peu de variantes dans le modèle.
- On n'a pas de machine macOS x86, donc on ne peut pas croiser complètement OS et processeur.
- On teste seulement deux propriétés.